# 当前正式知识pipeline：逐算子查看

清洗、筛选与图文联合提取后，仅一次final_review检查并写最终知识；程序检查并附回来源。每格显示真实Dataset，未合并历史多轮审核链。


**执行前请重启内核。** 禁止在已跑过旧代码的内核里逐个热替换类继续执行。代码、提示词或配置改变后使用新的RUN；相同版本才复用checkpoint。当前修改尚未重跑端到端，旧run原样保留。

### 当前正式执行链
原始 datasets → 文档清洗/过滤/去重、图片字节检查 → 概念身份及正文相关性 → **Qwen 图片初筛 → Gemma 独立复核含 keep 图的完整原批次 → 分歧暂缓** → 原生图文关联与相似度补充 → 按容量组装 → `joint_paragraphs` → `final_review` → 程序校验与保存。

联合提取仅收到概念范围、编号原文及必要上下文、真实图片；最后 review 只收到各组提取稿及其实际引用的原文和采用的图片。上游模型 caption 和筛选结论不作为证据。

两模型共用两张 GPU：阶段内部为 demiflow 流式处理，图片初筛和复核之间有完整 checkpoint 与服务切换。Gemma 完成后恢复原 Qwen 和预标注。`image_review_service='borrow'`管理借用恢复，`external`使用已准备好的外部服务。双模型一致不代表身份已经证实。


In [ ]:
from pathlib import Path
import sys, json, html
from IPython.display import display, HTML
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# 不对运行中的Dataset热替换算子：发现旧内核时停止，重启后再读取checkpoint。
def _assert_current_kernel():
    import os
    try:
        shell = get_ipython()
    except NameError:
        return  # CLI是独立Python进程。
    if shell is None or not hasattr(shell, 'kernel'):
        return
    boot = next(int(line.split()[1]) for line in Path('/proc/stat').read_text().splitlines() if line.startswith('btime '))
    started = boot + int(Path('/proc/self/stat').read_text().rsplit(')', 1)[1].split()[19]) / os.sysconf('SC_CLK_TCK')
    changed = []
    for name, module in tuple(sys.modules.items()):
        if name.startswith(('curation.', 'demiflow.')):
            filename = getattr(module, '__file__', None)
            if filename and Path(filename).is_file() and Path(filename).stat().st_mtime > started:
                changed.append(name)
    if changed:
        raise RuntimeError('当前内核启动后算子代码已更新，请重启内核并重新执行初始化；禁止新旧算子混用。涉及：' + ', '.join(changed[:6]))
_assert_current_kernel()
from curation.v4.ops.filter_document_blocks import FilterDocumentBlocks
from curation.v4.ops.select_source_records import SelectSourceRecords
from demiflow.standalone import local_data
from curation.v4.contracts import snapshot, immutable, digest, source_code, runtime_version
from curation.v4.pipeline import DEFAULT
from curation.v4.ops.image_filter import (IMAGE_FILTER_DEFAULTS, RecordPrimaryImageSelection, PrepareImageReview, ApplyConfirmedImageSelection)
from curation.v4.image_filter_runtime import image_prompt_data, review_needed, save_image_filter_policy, validate_material_reuse
from curation.v4.local_review_service import image_review_service
from curation.v4.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks,
    ReadDocument, CleanDocument, CheckImage, CountMaterial, NestMaterial,
    merge_concept, distinct, fill_material_counts, model_input)
from curation.v4.ops.prompt_operators import PrepareIdentity, ApplyIdentity
from curation.v4.ops.prompt_config import knowledge_prompt_pack, prompt_execution_options, save_prompt_config
from curation.v4.ops.source_blocks import BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection, merge_block_decisions
from curation.v4.ops.multimodal import SelectAvailableImages, BatchImageSelection, merge_image_decisions, SelectRelatedMaterials
from curation.v4.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.v4.ops.paragraph_similarity import EmbedParagraphBatch
from curation.v4.ops.token_routing import RouteByTokenBudget

DATASET = ROOT / 'datasets/demiwtg'
RUN = ROOT / 'state/curation/v4/glass_operator_article_v1'
CONCEPT = '玻璃棒'
IDS = ['legacy:' + CONCEPT]
GROUP_SIZE = 256
# 本轮三个概念的明确身份范围；扩量时从概念资料确定，勿按名称猜物种。
IMAGE_IDENTITY_DEFINITIONS = {'legacy:OK手势': '拇指和食指相触成环、其余手指伸展或放松的手势；相关图解和实际使用场景也可保留。', 'legacy:玻璃棒': '实验室中用于搅拌、引流等的实心玻璃棒；同属实验器材不自动属于目标。', 'legacy:白花芍药': '植物学物种 Paeonia sterniana；泛指白色芍药花或其他栽培品种不自动认证为这一物种。身份不确定时保留不确定性。'}
config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, 'image_identity_definitions': IMAGE_IDENTITY_DEFINITIONS, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None,
          'max_output_tokens':16384, 'temperature':0, 'timeout_s':900,
          'block_unit_chars':1800, 'block_batch_chars':8000,
          'article_mode':True,'joint_thinking':True,'joint_reasoning_effort':'low','final_review_thinking':True,'final_review_effort':'low','final_max_output_tokens':32768,'final_timeout_s':1200, 'final_input_tokens':131072, 'joint_input_tokens':32768, 'image_batch_size':4,
          'text_embedding_model':str(ROOT.parent / 'models/Qwen3-Embedding-0.6B'),
          'image_embedding_model':str(ROOT.parent / 'models/siglip2-base-patch16-224')}
tables = RUN / 'datasets'
knowledge_run = RUN / 'knowledge'

def show(ds, columns=None, n=100):
    from curation.notebook_image_preview import show as preview
    return preview(ds, columns=columns, n=n, run=RUN, dataset=DATASET)

from curation.v4.ops.article import PrepareArticleInput, ApplyArticle, PrepareFinalReview, PublishArticle, ArticleTokenBudget, PrepareSelectionScope, EnsureConceptLabel


### 冻结本次输入、代码和配置

In [ ]:
concept_source = {'kind':'legacy_concepts', **snapshot(DATASET / 'meta/concepts.json')}
document_source = {'kind':'legacy_docs', **snapshot(DATASET / 'meta/docs.jsonl')}
image_source = {'kind':'legacy_images', **snapshot(DATASET / 'meta/images.jsonl')}
notebook = json.loads((ROOT / 'curation/v4/glass_operator_debug.ipynb').read_text())
manifest = {'sources':[concept_source, document_source, image_source],
            'ids':IDS, 'group_size':GROUP_SIZE, 'config':config,
            'code':source_code(), 'runtime':runtime_version(),
            'cells':[''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code']}
from curation.notebook_image_preview import preserve_display_version
manifest = preserve_display_version(RUN, manifest)
immutable(RUN / 'manifest.json', manifest)
version = digest(manifest)
pack, prompt_text = knowledge_prompt_pack(config)
options = prompt_execution_options(RUN, config)
save_prompt_config(RUN, prompt_text, options)
data = local_data(prompt_packs={'knowledge.yaml':pack}, prompt_options=options)

# 每个后续步骤执行前核对运行中代码/依赖/配置，禁止沿用旧version写新结果。
import copy
_frozen_code, _frozen_runtime = source_code(), runtime_version()
_frozen_config, _frozen_run = copy.deepcopy(config), RUN

def _assert_run_current():
    _assert_current_kernel()
    if RUN != _frozen_run or config != _frozen_config or source_code() != _frozen_code or runtime_version() != _frozen_runtime:
        raise RuntimeError('本次冻结后代码、依赖、配置或RUN已变化；请重启内核并使用新RUN，不得混用旧checkpoint。')


### 1．read_records · 原始概念记录

In [ ]:
_assert_run_current()
concept_records = data.read_records(DATASET / 'meta/concepts.json', format='json', item_prefix='concepts.item',
    report_path=RUN / 'source_status/concepts.json').filter(SelectSourceRecords('legacy_concepts', IDS))
# 原生读取后下推概念筛选；仍扫描原清单，只有入选关联记录进入后续转换。
show(concept_records.filter(lambda r: r['error'] is None and isinstance(r['value'],dict) and r['value'].get('name') == CONCEPT), n=1)


### 2．ConceptFromRecord · 转换概念字段

In [ ]:
_assert_run_current()
concepts = concept_records.filter(lambda r: r['error'] is None and isinstance(r['value'],dict)).map(ConceptFromRecord(concept_source))
show(concepts.filter(lambda r:r['concept_ref'] in IDS), n=1)


### 3．SelectConcept · 筛选玻璃棒

In [ ]:
_assert_run_current()
selected_concepts = await concepts.map(SelectConcept(IDS)).filter(lambda r: r['selected']).reduce_by_key('concept_ref', merge_concept).checkpoint_async(tables / 'selected_concepts.jsonl', version=version)
show(selected_concepts, columns=['concept_ref', 'name', 'aliases', 'qid', 'source_records'])


### 4．read_records · 原始文档清单

In [ ]:
_assert_run_current()
document_records = data.read_records(DATASET / 'meta/docs.jsonl', report_path=RUN / 'source_status/documents.json').filter(SelectSourceRecords('legacy_docs', IDS))
show(document_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict) and CONCEPT in r['value'].get('concepts', [])), n=5)


### 5．DocumentFromRecord · 转换文档字段

In [ ]:
_assert_run_current()
documents = document_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict)).map(DocumentFromRecord(document_source))
show(documents.filter(lambda r: any(ref in IDS for ref in r['concept_refs'])), columns=['doc_id','title','url','path','concept_refs'])


### 6．MaterialLinks + join · 关联入选概念的文档

In [ ]:
_assert_run_current()
document_links = await documents.flat_map(MaterialLinks('doc_id')).join(selected_concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').checkpoint_async(tables / 'document_links.jsonl', version=version)
show(document_links)


### 7．join · 保留关联文档

In [ ]:
_assert_run_current()
selected_documents = await documents.join(document_links.select_columns(['doc_id']).reduce_by_key('doc_id', distinct), on='doc_id', how='semi').checkpoint_async(tables / 'selected_documents.jsonl', version=version)
show(selected_documents, columns=['doc_id', 'title', 'url', 'path'])


### 8．ReadDocument · 读取原始页面文本

In [ ]:
_assert_run_current()
raw_documents = await selected_documents.map_cached(ReadDocument(DATASET), cache_dir=RUN / 'cache/read_documents', version=version).checkpoint_async(tables / 'raw_documents.jsonl', version=version)
show(raw_documents, columns=['doc_id', 'title', 'url', 'raw_text', 'read_status', 'read_error'])


### 9．CleanDocument · 清洗页面正文

In [ ]:
_assert_run_current()
baseline_documents = await raw_documents.map_cached(CleanDocument(), cache_dir=RUN / 'cache/clean_documents', version=version).checkpoint_async(tables / 'baseline_documents.jsonl', version=version)
show(baseline_documents, columns=['doc_id', 'title', 'raw_text', 'clean_text', 'clean_blocks', 'clean_status'])


### 9b．FilterDocumentBlocks · 过滤非正文、段落去重、修复导航和引用标记
输入：上格解析的文档及原文块。输出：更新后的 clean_text、clean_blocks（重建位置）、clean_filter（修改依据及未解决字段）。原文与图片关联保留；不运行概念相关性判断，不猜测断裂字段配对。


In [ ]:
_assert_run_current()
processed_documents = await baseline_documents.map_cached(FilterDocumentBlocks(), cache_dir=RUN / 'cache/filter_documents', version=version).checkpoint_async(tables / 'processed_documents.jsonl', version=version)
show(processed_documents, columns=['doc_id', 'title', 'clean_text', 'clean_blocks', 'clean_filter', 'knowledge_eligibility'])


### 10．read_records · 原始图片清单

In [ ]:
_assert_run_current()
image_records = data.read_records(DATASET / 'meta/images.jsonl', report_path=RUN / 'source_status/images.json').filter(SelectSourceRecords('legacy_images', IDS))
show(image_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict) and CONCEPT in (r['value'].get('concepts') or r['value'].get('instances') or [])), n=5)


### 11．ImageFromRecord · 转换图片字段

In [ ]:
_assert_run_current()
images = image_records.filter(lambda r:r['error'] is None and isinstance(r['value'],dict)).map(ImageFromRecord(image_source))
show(images.filter(lambda r:any(ref in IDS for ref in r['concept_refs'])), columns=['image_id','path','caption','concept_refs'], n=5)


### 12．MaterialLinks + join · 关联入选概念的图片

In [ ]:
_assert_run_current()
image_links = await images.flat_map(MaterialLinks('image_id')).join(selected_concepts.select_columns(['concept_ref']), on='concept_ref', how='semi').checkpoint_async(tables / 'image_links.jsonl', version=version)
show(image_links)


### 13．join · 保留关联图片

In [ ]:
_assert_run_current()
selected_images = await images.join(image_links.select_columns(['image_id']).reduce_by_key('image_id', distinct), on='image_id', how='semi').checkpoint_async(tables / 'selected_images.jsonl', version=version)
show(selected_images, columns=['image_id', 'path', 'caption', 'content_url', 'landing_url'])


### 14．CheckImage · 检查实际图片文件

In [ ]:
_assert_run_current()
processed_images = await selected_images.map_cached(CheckImage(DATASET), cache_dir=RUN / 'cache/check_images', version=version).checkpoint_async(tables / 'processed_images.jsonl', version=version)
show(processed_images, columns=['image_id', 'path', 'byte_status', 'byte_details'])


### 15．CountMaterial · 文档数量和读取状态

In [ ]:
_assert_run_current()
document_counts = await document_links.join(processed_documents.select_columns(['doc_id', 'read_status']), on='doc_id').reduce_by_key('concept_ref', CountMaterial('document_count', 'read_status', 'readable_documents')).checkpoint_async(tables / 'document_counts.jsonl', version=version)
show(document_counts)


### 16．CountMaterial · 图片数量和文件状态

In [ ]:
_assert_run_current()
image_counts = await image_links.join(processed_images.select_columns(['image_id', 'byte_status']), on='image_id').reduce_by_key('concept_ref', CountMaterial('image_count', 'byte_status', 'verified_images')).checkpoint_async(tables / 'image_counts.jsonl', version=version)
show(image_counts)


### 17．join · 将资料数量关联到概念

In [ ]:
_assert_run_current()
concepts_ready = await selected_concepts.join(document_counts, on='concept_ref', how='left').join(image_counts, on='concept_ref', how='left').map(fill_material_counts).checkpoint_async(tables / 'concepts_ready.jsonl', version=version)
show(concepts_ready, columns=['concept_ref', 'name', 'document_count', 'readable_documents', 'image_count', 'verified_images'])


### 18．NestMaterial · 文档按概念关联

In [ ]:
_assert_run_current()
concept_documents = await document_links.join(processed_documents.map(NestMaterial('doc_id', 'documents')), on='doc_id').checkpoint_async(tables / 'concept_documents.jsonl', version=version)
show(concept_documents)


### 19．NestMaterial · 图片按概念关联

In [ ]:
_assert_run_current()
concept_images = await image_links.join(processed_images.map(NestMaterial('image_id', 'images')), on='image_id').checkpoint_async(tables / 'concept_images.jsonl', version=version)
show(concept_images)


### 20．group_batches · 按概念汇集资料

In [ ]:
_assert_run_current()
material_batches = await concept_documents.union(concept_images).group_batches('concept_ref', max_rows=GROUP_SIZE, output='materials').checkpoint_async(tables / 'material_batches.jsonl', version=version)
show(material_batches)


### 21．join · 概念与资料批次

In [ ]:
_assert_run_current()
batches = await concepts_ready.join(material_batches, on='concept_ref', how='left').checkpoint_async(tables / 'batches.jsonl', version=version)
show(batches, columns=['concept_ref', 'name', 'materials'])


### 22．model_input · 转成现有模型算子的输入

In [ ]:
_assert_run_current()
model_inputs = await batches.map(model_input).checkpoint_async(tables / 'model_inputs.jsonl', version=version)
show(model_inputs)


### 23．PrepareIdentity · 准备身份判断输入

In [ ]:
_assert_run_current()
identity_inputs = await model_inputs.map_cached(PrepareIdentity(knowledge_run, config), cache_dir=RUN / 'cache/identity_prepare', version=version).checkpoint_async(tables / 'identity_inputs.jsonl', version=version)
show(identity_inputs, columns=['case_id', 'identity_prompt'])


### 24．identity · 模型调用

In [ ]:
_assert_run_current()
identity_responses = await identity_inputs.map_prompt_async('identity', config='knowledge.yaml', inputs={'payload': 'identity_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1, when=lambda r: not r.get('blocked') and 'identity_prompt' in r).checkpoint_async(tables / 'identity_responses.jsonl', version=version)
show(identity_responses, columns=['prompt_result', 'prompt_error'])


### 25．ApplyIdentity · 解析身份判断

In [ ]:
_assert_run_current()
identified = await identity_responses.map_cached(ApplyIdentity(knowledge_run, config), cache_dir=RUN / 'cache/identity_apply', version=version).checkpoint_async(tables / 'identified.jsonl', version=version)
show(identified, columns=['case_id', 'identity', 'identity_materials', 'identity_unexamined'])


### 26．BuildSourceBlocks · 按原文章节拆正文块

In [ ]:
_assert_run_current()
source_blocks = await identified.map(EnsureConceptLabel()).map(BuildSourceBlocks(config['block_unit_chars'], body_only=True)).checkpoint_async(tables / 'source_blocks.jsonl', version=version)
show(source_blocks, columns=['case_id', 'source_units'])


### 27．SelectAvailableImages · 收集文件可用的图片

In [ ]:
_assert_run_current()
blocks = await source_blocks.map(SelectAvailableImages()).checkpoint_async(tables / 'blocks.jsonl', version=version)
show(blocks, columns=['case_id', 'available_images', 'image_material_scope'])


### 28．BatchSourceBlocks · 组装文字筛选请求

In [ ]:
_assert_run_current()
text_requests = await blocks.flat_map(BatchSourceBlocks(config['block_batch_chars'])).map(PrepareSelectionScope(config['image_identity_definitions'])).checkpoint_async(tables / 'text_requests.jsonl', version=version)
show(text_requests, columns=['case_id', 'block_prompt'])


### 29．select_blocks · 模型调用

In [ ]:
_assert_run_current()
text_responses = await text_requests.map_prompt_async('select_blocks', config='knowledge.yaml', inputs={'payload': 'block_prompt'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'text_responses.jsonl', version=version)
show(text_responses, columns=['prompt_result', 'prompt_error'])


### 30．ApplyBlockSelection · 解析正文相关性判断

In [ ]:
_assert_run_current()
text_decisions = await text_responses.map_cached(ApplyBlockSelection(relevance_only=True), cache_dir=RUN / 'cache/text_selection', version=version).checkpoint_async(tables / 'text_decisions.jsonl', version=version)
show(text_decisions)


### 31．reduce_by_key · 汇集正文筛选结果

In [ ]:
_assert_run_current()
text_by_concept = await text_decisions.reduce_by_key('case_id', merge_block_decisions).checkpoint_async(tables / 'text_by_concept.jsonl', version=version)
show(text_by_concept)


### 32．BatchImageSelection · 组装图片筛选请求

In [ ]:
_assert_run_current()
image_requests = await blocks.flat_map(BatchImageSelection(config['image_batch_size'], config['image_identity_definitions'], neutral=True)).checkpoint_async(tables / 'image_requests.jsonl', version=version)
show(image_requests, columns=['case_id', 'image_prompt', 'pixel_images'])


### 33．select_images · Qwen3.8图片初筛

In [ ]:
_assert_run_current()
primary_data = image_prompt_data(RUN, config)
image_responses = await primary_data.read_json(str(tables / 'image_requests.jsonl')).map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload': 'image_prompt', 'images': 'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=1, queue_depth=1).checkpoint_async(tables / 'image_responses.jsonl', version=version)
show(image_responses, columns=['prompt_result', 'prompt_error'])


### 34．RecordPrimaryImageSelection → PrepareImageReview · 保存初筛、准备独立复核

In [ ]:
_assert_run_current()
# 输入：Qwen响应及原批次像素。输出：初筛判断及独立复核请求；最终判断见34C。
primary = await image_responses.map_cached(RecordPrimaryImageSelection(), cache_dir=RUN / 'cache/image_primary', version=version).checkpoint_async(tables / 'image_primary.jsonl', version=version)
review_rows = await primary.map(PrepareImageReview()).checkpoint_async(tables / 'image_review_inputs.jsonl', version=version)
show(review_rows, columns=['image_prompt', 'review_required', 'primary_selection'])

### 34B．Gemma独立复核入选图
输入为原批次相同图文，模型不会收到Qwen判断。只有含初筛入选图的批次调用Gemma。无可用8001服务时自动让预标注落盘、借用GPU；完成或失败均恢复原Qwen。已完成checkpoint复用时不切服务。

In [ ]:
_assert_run_current()
review_data = image_prompt_data(RUN, config, review=True)
review_requests = review_data.read_json(str(tables / 'image_review_inputs.jsonl')).filter(lambda r: r['review_required'])
review_path = tables / 'image_review_responses.jsonl'
with image_review_service(RUN, config, needed=review_needed(review_requests, review_path, version)):
    reviewed = await review_requests.map_prompt_async('select_images', config='knowledge.yaml', inputs={'payload':'image_prompt','images':'pixel_images'}, output='prompt_result', call_output='prompt_call', error_output='prompt_error', concurrency=config['image_review_concurrency'], queue_depth=config['image_review_concurrency']).checkpoint_async(review_path, version=version)
show(reviewed, columns=['image_prompt', 'prompt_result', 'prompt_error'])

### 34C．ApplyConfirmedImageSelection · 合并两次独立判断
输入为Qwen初筛及Gemma复核。原排除／待定继续保留；Qwen入选且Gemma也入选才进入联合提炼，分歧或复核失败暂缓。输出保留两份原始观察与请求定位，不把一致当成身份认证。

In [ ]:
_assert_run_current()
image_decisions = await reviewed.union(review_rows.filter(lambda r: not r['review_required'])).map_cached(ApplyConfirmedImageSelection(), cache_dir=RUN / 'cache/image_confirmed', version=version).checkpoint_async(tables / 'image_decisions.jsonl', version=version)
show(image_decisions, columns=['case_id', 'image_decisions'])

### 35．reduce_by_key · 汇集图片筛选结果

In [ ]:
_assert_run_current()
image_by_concept = await image_decisions.reduce_by_key('case_id', merge_image_decisions).checkpoint_async(tables / 'image_by_concept.jsonl', version=version)
show(image_by_concept)


### 36．SelectRelatedMaterials · 保留相关原文和图片

In [ ]:
_assert_run_current()
related = await blocks.join(text_by_concept, on='case_id', how='left').join(image_by_concept, on='case_id', how='left').map(SelectRelatedMaterials()).checkpoint_async(tables / 'related.jsonl', version=version)
show(related, columns=['case_id', 'material_pack'])

save_image_filter_policy(RUN, config)

### 37．PrepareRoutingMaterials · 原生图文关联

In [ ]:
_assert_run_current()
routing_materials = await related.map(PrepareRoutingMaterials()).checkpoint_async(tables / 'routing_materials.jsonl', version=version)
show(routing_materials, columns=['concept', 'passages', 'images', 'native_links', 'unmatched_native_references'])


### 38．RawPassageRows · 展开正文块

In [ ]:
_assert_run_current()
passage_rows = await routing_materials.flat_map(RawPassageRows()).checkpoint_async(tables / 'passage_rows.jsonl', version=version)
show(passage_rows)


### 39．EmbedParagraphBatch · 正文向量

In [ ]:
_assert_run_current()
text_embeddings = await passage_rows.group_batches('embedding_bucket', max_rows=2, output='items').map_cached(EmbedParagraphBatch(config['text_embedding_model']), cache_dir=RUN / 'cache/input_embeddings', version=version).checkpoint_async(tables / 'text_embeddings.jsonl', version=version)
show(text_embeddings)


### 40．reduce_by_key · 按概念汇集正文向量

In [ ]:
_assert_run_current()
text_vectors = await text_embeddings.flat_map(lambda r: r['items']).reduce_by_key('case_id', lambda a, r: {'case_id': r['case_id'], 'passage_embeddings': {**a['passage_embeddings'], r['source_id']: r}}, initial={'passage_embeddings': {}}).checkpoint_async(tables / 'text_vectors.jsonl', version=version)
show(text_vectors)


### 41．EncodeImageTextMaterials · 图文相似度所需向量

In [ ]:
_assert_run_current()
image_text_vectors = await routing_materials.map_cached(EncodeImageTextMaterials(config['image_embedding_model']), cache_dir=RUN / 'cache/input_image_embeddings', version=version).checkpoint_async(tables / 'image_text_vectors.jsonl', version=version)
show(image_text_vectors)


### 42．RouteByTokenBudget · 按32K输入容量装组，能整组则不拆分

In [ ]:
_assert_run_current()
routed = await routing_materials.join(text_vectors, on='case_id', how='left').join(image_text_vectors.select_columns(['case_id', 'text_windows', 'image_vectors']), on='case_id', how='left').map(RouteByTokenBudget(ROOT.parent/'models/Qwen3.8-27B', config.get('joint_input_tokens',32768), counter=ArticleTokenBudget(ROOT.parent/'models/Qwen3.8-27B', {**config,'enable_thinking':config.get('joint_thinking',True),'reasoning_effort':config.get('joint_reasoning_effort','low')}, 'joint_paragraphs'))).checkpoint_async(tables / 'routed.jsonl', version=version)
show(routed, columns=['concept', 'requests', 'edges', 'overflow_edges'])


### 43．BuildRoutedJointRequest · 联合提炼的实际输入

In [ ]:
_assert_run_current()
joint_requests = await routed.flat_map(lambda r: r['requests']).map(BuildRoutedJointRequest()).checkpoint_async(tables / 'joint_requests.jsonl', version=version)
show(joint_requests, columns=['batch_id', 'joint_prompt', 'pixel_images'])


## 联合提取输入：概念、原文、真实图片


In [ ]:
_assert_run_current()
joint_config={**config,'enable_thinking':config.get('joint_thinking',True),'reasoning_effort':config.get('joint_reasoning_effort','low')}
joint_options=prompt_execution_options(RUN,joint_config)
save_prompt_config(RUN/'joint_extraction',prompt_text,joint_options)
joint_data=local_data(prompt_packs={'knowledge.yaml':pack},prompt_options=joint_options,max_prompt_requests=config['max_calls'])
article_inputs = await joint_data.read_json(str(tables/'joint_requests.jsonl')).map(PrepareArticleInput(config['image_identity_definitions'])).checkpoint_async(tables/'article_inputs.jsonl',version=version)
show(article_inputs, columns=['concept','article_input'])

## joint_paragraphs：提取图文知识


In [ ]:
_assert_run_current()
joint_responses = await article_inputs.map_prompt_async('joint_paragraphs',config='knowledge.yaml',inputs={'payload':'article_input','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1).checkpoint_async(tables/'joint_responses.jsonl',version=version)
show(joint_responses, columns=['batch_id','prompt_result','prompt_error'])

## 程序检查提取稿格式和引用


In [ ]:
_assert_run_current()
extracted = await joint_responses.map(ApplyArticle()).checkpoint_async(tables/'paragraph_extract.jsonl',version=version)
show(extracted, columns=['batch_id','article_status','validation_issues','article_text'])

## 收齐同概念各组提取结果


In [ ]:
_assert_run_current()
draft_groups = await extracted.reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'drafts':acc['drafts']+[r]},initial={'drafts':[]}).checkpoint_async(tables/'draft_groups.jsonl',version=version)
show(draft_groups)

## 只附回被引用的原文和图片，检查容量


In [ ]:
_assert_run_current()
review_config={**config,'enable_thinking':config.get('final_review_thinking',True),'reasoning_effort':config.get('final_review_effort','low'), 'max_output_tokens':config.get('final_max_output_tokens',32768),'timeout_s':config.get('final_timeout_s',1200)}
review_options=prompt_execution_options(RUN,review_config)
save_prompt_config(RUN/'final_review',prompt_text,review_options)
review_data=local_data(prompt_packs={'knowledge.yaml':pack},prompt_options=review_options,max_prompt_requests=config['max_calls'])
review_requests = await draft_groups.map(PrepareFinalReview(config['image_identity_definitions'],ArticleTokenBudget(ROOT.parent/'models/Qwen3.8-27B',review_config,'final_review'),config.get('final_input_tokens',131072))).checkpoint_async(tables/'final_review_requests.jsonl',version=version)
show(review_requests, columns=['concept','parent_batches','preflight_error','input_token_budget','article_input'])

## final_review：一次review并写最终知识


In [ ]:
_assert_run_current()
review_responses = await review_data.read_json(str(tables/'final_review_requests.jsonl')).map_prompt_async('final_review',config='knowledge.yaml',inputs={'payload':'article_input','images':'pixel_images'},when=lambda r:not r['preflight_error'],output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1).checkpoint_async(tables/'review_responses.jsonl',version=version)
show(review_responses, columns=['concept','prompt_result','prompt_error'])

## 程序检查最终文章


In [ ]:
_assert_run_current()
reviewed = await review_responses.map(ApplyArticle(final=True)).checkpoint_async(tables/'final_review.jsonl',version=version)
show(reviewed, columns=['concept','article_status','validation_issues','article_text'])

## 附回来源与图片，保存最终知识和状态


In [ ]:
_assert_run_current()
material_groups = related.map(lambda r:{'concept':r['identity']['target_label'],'material':r}).reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'materials':acc['materials']+[r['material']]},initial={'materials':[]})
final = await material_groups.join(reviewed.map(lambda r:{'concept':r['concept'],'review':r}),on='concept',how='left').map(PublishArticle()).checkpoint_async(RUN/'knowledge_base.jsonl',version=version)
show(final, columns=['concept','status','status_reason','knowledge'])

## 展示当前运行的最终结果


In [ ]:
_assert_run_current()
import importlib
from curation.v4 import current_results
importlib.reload(current_results)
current_results.show_current_results(RUN, concepts=[CONCEPT], limit=None, images=True)